# JEPA trajectoires Porto — entraînement GPU (Colab)

Code cloné depuis GitHub, données lues depuis Google Drive. Le code est
**device-agnostic** : le GPU CUDA de Colab est détecté automatiquement.

**Prérequis (une fois, en local puis sur Drive) :**
1. Générer le cache : `python prepare_data.py --limit 200000`
2. Déposer `data/trips.npz` et `data/feat_stats.npz` dans
   `MyDrive/jepa-taxi/data/` sur ton Google Drive.

**Runtime :** *Exécution → Modifier le type d'exécution → GPU*.

In [ ]:
!nvidia-smi -L
import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# 1) Cloner le code depuis GitHub
REPO_URL = "https://github.com/<user>/<repo>.git"   # <-- mets l'URL de TON repo
!git clone $REPO_URL jepa-taxi
%cd jepa-taxi
!git log --oneline -1

In [ ]:
# 2) Dépendances (torch déjà présent sur Colab)
!pip -q install scikit-learn matplotlib numpy

In [ ]:
# 3) Monter Google Drive (cache de données)
from google.colab import drive
drive.mount('/content/drive')
DATA = "/content/drive/MyDrive/jepa-taxi/data"
import os
print('trips.npz     :', os.path.exists(os.path.join(DATA, 'trips.npz')))
print('feat_stats.npz:', os.path.exists(os.path.join(DATA, 'feat_stats.npz')))

In [ ]:
# 4) Sanity overfit (rapide) — vérifie que ça apprend sans collapse
!python -m jepa.train --overfit \
    --data-path /content/drive/MyDrive/jepa-taxi/data/trips.npz --feat-stats /content/drive/MyDrive/jepa-taxi/data/feat_stats.npz

In [ ]:
# 5) Entraînement sur GPU (ajuste subset/epochs/batch selon le GPU)
!python -m jepa.train --subset 200000 --epochs 20 --batch-size 256 \
    --data-path /content/drive/MyDrive/jepa-taxi/data/trips.npz --feat-stats /content/drive/MyDrive/jepa-taxi/data/feat_stats.npz

In [ ]:
# 6) Évaluation (4 protocoles) + figures
!python -m jepa.eval --ckpt checkpoints/jepa.pt --n-eval 8000 \
    --data-path /content/drive/MyDrive/jepa-taxi/data/trips.npz --feat-stats /content/drive/MyDrive/jepa-taxi/data/feat_stats.npz
from IPython.display import Image
Image('eval_out/clusters_map.png')

In [ ]:
# 7) Sauvegarder le checkpoint sur le Drive (persistant entre sessions)
import shutil, os
os.makedirs('/content/drive/MyDrive/jepa-taxi/checkpoints', exist_ok=True)
shutil.copy('checkpoints/jepa.pt', '/content/drive/MyDrive/jepa-taxi/checkpoints/')
print('checkpoint copié sur le Drive')

### Notes
- Tous les hyperparamètres sont dans `jepa/config.py` ; sur GPU on peut monter
  `d_model`, `n_layers`, `batch_size`, `subset_size` (voire tout le dataset).
- Pour re-tirer les dernières modifs du code : `!git pull` dans le dossier cloné.
- `train.csv` (2 Go) n'est **pas** nécessaire sur Colab.